# Sakana-Agent-Forge 3B Agent Distillation and Self-Play Training Pipeline

This notebook provides the specification to run the distillation and self-play training on Google Colab (requires GPU, e.g., T4, L4, or A100).

## 1. Environment Setup

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

## 2. Load the scripts
Assuming you've uploaded `distill_unsloth.py` and `self_play_verifier.py` to your Colab workspace.

In [ ]:
import distill_unsloth
import self_play_verifier

## 3. Generate Distillation Data via Self-Play
We use the verifier to score the generated outputs and create a dataset for the student.

In [ ]:
import json

verifier = self_play_verifier.RewardVerifier()

# In a real setup, we would generate these responses from a Teacher model (e.g., Llama-3-70B)
# and filter them using the verifier.
dummy_teacher_outputs = [
    {
        "instruction": "Solve the following task using tools.",
        "input": "User: What is 2+2?",
        "output": '```json\n{"tool": "calculator", "args": {"expression": "2+2"}}\n```'
    },
    {
        "instruction": "Solve the following task using tools.",
        "input": "User: Search for Sakana AI.",
        "output": '```json\n{"tool": "search", "args": {"query": "Sakana AI"}}\n```'
    }
]

filtered_data = []
for data in dummy_teacher_outputs:
    # Evaluate the step using the verifier
    reward = verifier.evaluate_step(data["output"])
    if reward > 0:
        filtered_data.append(data)

print(f"Generated {len(filtered_data)} high-quality samples from self-play/teacher.")

with open("teacher_student_data.jsonl", "w") as f:
    for d in filtered_data:
        f.write(json.dumps(d) + "\n")

## 4. Run Distillation using Unsloth (Student Training)
We use Unsloth to train a 3B model (e.g., Llama-3.2-3B) with QLoRA on the generated dataset.

In [ ]:
import os
os.environ["DATASET_PATH"] = "teacher_student_data.jsonl"

distill_unsloth.main()

## 5. Inference / Verification
Test the newly trained student model.

In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "distilled_lora_model", # Load the saved model
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

prompt_template = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
"""

inputs = tokenizer(
[
    prompt_template.format(
        "Solve the following task using tools.",
        "User: Calculate 10 * 5."
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True)
print(tokenizer.batch_decode(outputs))